# Analysis Pipeline: Usability of XAI Interfaces

This notebook executes the full statistical analysis pipeline for the usability study.
It includes data loading, cleaning, statistical testing (ANOVA), visualization, and reporting.

**Reproducibility**: All random seeds are set at the beginning.

In [ ]:
# Setup: Seeds and Imports
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Ensure data directories exist
project_root = Path.cwd()
data_dir = project_root / "data" / "processed"
data_dir.mkdir(parents=True, exist_ok=True)

# Add code to path if needed for imports
code_dir = project_root / "code"
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

print(f"Project root: {project_root}")
print(f"Data directory: {data_dir}")

## 1. Data Loading and Cleaning

Load raw session data, apply exclusion criteria (incomplete sessions), and prepare for analysis.

In [ ]:
# Import data cleaning module
from analysis.data_cleaner import DataCleaner

# Initialize cleaner
cleaner = DataCleaner()

# Run cleaning pipeline
# This loads from data/raw/*.json, filters incomplete sessions, and saves to data/processed/cleaned_sessions.csv
cleaned_df = cleaner.run_pipeline(
    raw_dir=project_root / "data" / "raw",
    output_path=data_dir / "cleaned_sessions.csv"
)

print(f"Cleaned data shape: {cleaned_df.shape}")
print(cleaned_df.head())

## 2. Statistical Analysis

Perform Repeated Measures ANOVA on Completion Time, Error Count, and SUS.
Exclude `explanation_engagement_time` from inferential testing.

In [ ]:
# Import statistical utilities
from analysis.stat_utils import run_anova_pipeline, generate_metrics_summary

# Run ANOVA pipeline
# This generates metrics_summary.csv
anova_results = run_anova_pipeline(
    df=cleaned_df,
    metrics=["completion_time", "error_count", "sus_score"],
    output_path=data_dir / "metrics_summary.csv"
)

print("ANOVA Results Summary:")
print(anova_results)

In [ ]:
# Load and display the generated metrics summary
metrics_df = pd.read_csv(data_dir / "metrics_summary.csv")
print(metrics_df)

## 3. Descriptive Statistics for Engagement Time

Report descriptive stats for `explanation_engagement_time` (descriptive only, not inferential).

In [ ]:
# Import descriptive stats runner
from analysis.run_descriptive_stats import load_raw_session_data, main as desc_main

# Run descriptive stats
desc_stats = desc_main(
    raw_dir=project_root / "data" / "raw",
    output_path=data_dir / "descriptive_stats.csv"
)

print("Descriptive Stats for Engagement Time:")
print(desc_stats)

## 4. Visualization

Generate box plots with error bars for the significant metrics.

In [ ]:
# Import visualizer
from analysis.visualizer import Visualizer

viz = Visualizer()

# Generate plots
viz.plot_boxplots(
    df=cleaned_df,
    metrics=["completion_time", "error_count", "sus_score"],
    x_col="interface_type",
    output_dir=data_dir
)

print(f"Plots saved to {data_dir}")

## 5. Power Analysis and Limitations

Compute statistical power and identify underpowered subgroups.

In [ ]:
# Import power analysis
from analysis.power_analysis import PowerCalculator

power_calc = PowerCalculator()

# Run power analysis
power_flags = power_calc.run_power_analysis(
    df=cleaned_df,
    metrics=["completion_time", "error_count", "sus_score"],
    output_path=data_dir / "power_flags.json"
)

print("Power Analysis Flags:")
for flag in power_flags:
    print(f"  - {flag}")

## 6. Report Generation

Generate the final summary report including N, power flags, and exclusion reasons.

In [ ]:
# Import report generator
from analysis.report_generator import ReportGenerator

report_gen = ReportGenerator()

# Generate report
report_path = data_dir / "report_summary.txt"
report_gen.generate_report(
    cleaned_df=cleaned_df,
    metrics_df=metrics_df,
    power_flags=power_flags,
    output_path=report_path
)

print(f"Report generated at {report_path}")

# Display report content
with open(report_path, "r") as f:
    print(f.read())

## 7. Validation

Verify that all expected outputs were generated correctly.

In [ ]:
# Run validation script
import subprocess

validation_script = project_root / "code" / "analysis" / "validate_notebook_execution.py"
result = subprocess.run(
    [sys.executable, str(validation_script)],
    cwd=str(project_root)
)

if result.returncode == 0:
    print("\n✅ Validation PASSED. All outputs verified.")
else:
    print("\n❌ Validation FAILED. Check logs for details.")